# 1: Data Collection and Filtering

This notebook builds the dataset used for analysis across the rest of the project.


## 1.1 Setup and working directories

This section sets up the project root, the data folders, and the output folders so the rest of the notebook can read and write files consistently.


In [28]:
from __future__ import annotations

import csv
import io
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
# Resolve the repository root whether the notebook is run inside notebooks/ or from the project root.
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
# Raw monthly extracts and their processed derivatives both live under data/.
DATA_DIR = ROOT / "data"
# Keep filtered CSV outputs with train rows only in a stable location for the downstream notebooks.
PROCESSED_DIR = DATA_DIR / "processed"
# Store the hourly Open-Meteo exports separately from the train tables.
WEATHER_DIR = DATA_DIR / "weather"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
WEATHER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Data directory: {DATA_DIR.relative_to(ROOT)}")
print(f"Processed output dir: {PROCESSED_DIR.relative_to(ROOT)}")


Project root: /Users/berenaydogan/Documents/4.2/DSA210/Project/DSA210-Project
Data directory: data
Processed output dir: data/processed


## 1.2 Archive constants and date helpers

This section defines the sampled date ranges, archive naming rules, and the small helper functions that support the collection logic.

The sampled date ranges are:

- January 1 to January 30, 2025
- April 1 to April 30, 2025
- July 1 to July 30, 2025
- October 1 to October 30, 2025

I use one 30 day period from each season so the dataset covers winter, spring, summer, and autumn without requiring a full year of raw data.


In [29]:
import datetime as dt
import re
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from collections import defaultdict

ARCHIVE_INDEX_URL = "https://archive.opentransportdata.swiss/istdaten.php"
ARCHIVE_BASE_URL = "https://archive.opentransportdata.swiss/"
# Stream large monthly ZIP files in 1 MB chunks so the download can report progress.
CHUNK_SIZE = 1024 * 1024
# Use one shared timeout for every archive and weather HTTP request.
DOWNLOAD_TIMEOUT_SECONDS = 60
# Retry transient network failures a few times before giving up.
MAX_DOWNLOAD_RETRIES = 5
# Print progress periodically instead of on every chunk write.
PROGRESS_INTERVAL_SECONDS = 5
DEFAULT_SAMPLE_RANGES = (
    ("2025-01-01", "2025-01-30"),
    ("2025-04-01", "2025-04-30"),
    ("2025-07-01", "2025-07-30"),
    ("2025-10-01", "2025-10-30"),
)
# Find every monthly ZIP link published on the official archive index page.
ARCHIVE_LINK_RE = re.compile(r'href="(?P<href>istdaten/[^"]+\.zip)"')
# Match the modern archive naming scheme (for example 2025-01.zip).
ARCHIVE_MONTH_RE = re.compile(r"(?P<year>20\d{2})-(?P<month>\d{2})\.zip$")
# Match older short names that still appear in some archive listings.
ARCHIVE_MONTH_SHORT_RE = re.compile(r"(?P<yy>\d{2})_(?P<month>\d{2})\.zip$")
# Prefer the highest explicit archive version when multiple ZIPs exist for one month.
ARCHIVE_VERSION_RE = re.compile(r"-v(?P<version>\d+)-")


def parse_iso_date(raw: str) -> dt.date:
    """Convert a YYYY-MM-DD string into a Python date object."""
    return dt.date.fromisoformat(raw)


def expand_date_ranges(date_ranges: tuple[tuple[str, str], ...] = DEFAULT_SAMPLE_RANGES) -> list[dt.date]:
    """Expand inclusive date ranges like 2025-01-01..2025-01-30 into daily dates."""
    dates: list[dt.date] = []
    for raw_start, raw_end in date_ranges:
        start = parse_iso_date(raw_start)
        end = parse_iso_date(raw_end)
        if end < start:
            raise ValueError(f"Invalid range: {raw_start} > {raw_end}")
        current = start
        while current <= end:
            dates.append(current)
            current += dt.timedelta(days=1)
    return dates


def fetch_text(url: str) -> str:
    """Fetch a small UTF-8 text payload such as the archive index HTML page."""
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; DSA210 notebook downloader)"},
    )
    with urllib.request.urlopen(request, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response:
        return response.read().decode("utf-8")


def month_key_from_href(href: str) -> str | None:
    """Normalize an archive filename into a YYYY-MM key for month-level lookup."""
    name = Path(href).name
    match = ARCHIVE_MONTH_RE.search(name)
    if match:
        return f"{match.group('year')}-{match.group('month')}"

    match = ARCHIVE_MONTH_SHORT_RE.search(name)
    if match:
        return f"20{match.group('yy')}-{match.group('month')}"

    return None


def archive_version_rank(href: str) -> int:
    match = ARCHIVE_VERSION_RE.search(Path(href).name)
    if match:
        return int(match.group("version"))
    return 1


def parse_archive_index(html_text: str) -> dict[str, list[str]]:
    month_to_hrefs: dict[str, list[str]] = defaultdict(list)
    for match in ARCHIVE_LINK_RE.finditer(html_text):
        href = match.group("href")
        month_key = month_key_from_href(href)
        if month_key:
            month_to_hrefs[month_key].append(href)

    resolved: dict[str, list[str]] = {}
    for month_key, hrefs in month_to_hrefs.items():
        unique_hrefs = sorted(
            set(hrefs),
            key=lambda href: (archive_version_rank(href), href),
            reverse=True,
        )
        resolved[month_key] = [
            urllib.parse.urljoin(ARCHIVE_BASE_URL, href) for href in unique_hrefs
        ]
    return resolved


def group_dates_by_month(dates: list[dt.date]) -> dict[str, list[dt.date]]:
    grouped: dict[str, list[dt.date]] = defaultdict(list)
    for item in dates:
        grouped[item.strftime("%Y-%m")].append(item)
    return dict(grouped)


def format_bytes(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{int(value)} {unit}" if unit == "B" else f"{value:.1f} {unit}"
        value /= 1024
    return f"{num_bytes} B"


## 1.3 Archive index, download, and extraction helpers

This section collects the helper functions for finding the right monthly archives, downloading them when needed, and extracting only the daily CSV files that belong to the study period.


In [30]:
def head_file_size(url: str) -> int | None:
    request = urllib.request.Request(
        url,
        method="HEAD",
        headers={"User-Agent": "Mozilla/5.0 (compatible; DSA210 notebook downloader)"},
    )
    with urllib.request.urlopen(request, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response:
        length = response.headers.get("Content-Length")
        return int(length) if length is not None else None


def print_progress(downloaded: int, total: int | None, start_time: float) -> None:
    elapsed = max(time.monotonic() - start_time, 1e-9)
    speed = downloaded / elapsed
    if total:
        pct = downloaded / total * 100
        print(
            f"    progress: {format_bytes(downloaded)} / {format_bytes(total)} ({pct:5.1f}%) at {format_bytes(int(speed))}/s",
            flush=True,
        )
    else:
        print(f"    progress: {format_bytes(downloaded)} at {format_bytes(int(speed))}/s", flush=True)


def download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp_path = destination.with_suffix(destination.suffix + ".part")
    total_size = head_file_size(url)
    existing_size = temp_path.stat().st_size if temp_path.exists() else 0

    if total_size is not None and existing_size >= total_size:
        temp_path.replace(destination)
        return

    if existing_size:
        print(f"  resuming partial archive at {format_bytes(existing_size)}", flush=True)
    else:
        size_text = format_bytes(total_size) if total_size else "unknown size"
        print(f"  starting archive download ({size_text})", flush=True)

    downloaded = existing_size
    start_time = time.monotonic()
    last_progress = start_time
    retries = 0

    while True:
        headers = {"User-Agent": "Mozilla/5.0 (compatible; DSA210 notebook downloader)"}
        if downloaded:
            headers["Range"] = f"bytes={downloaded}-"
        request = urllib.request.Request(url, headers=headers)

        try:
            with urllib.request.urlopen(request, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response:
                status = response.getcode()
                # HTTP 206 confirms the server honored the resume request for a partial archive.
                if downloaded and status != 206:
                    temp_path.unlink(missing_ok=True)
                    downloaded = 0
                    print("  server did not honor resume request; restarting download", flush=True)
                    continue

                mode = "ab" if downloaded else "wb"
                with temp_path.open(mode) as fh:
                    while True:
                        chunk = response.read(CHUNK_SIZE)
                        if not chunk:
                            break
                        fh.write(chunk)
                        downloaded += len(chunk)
                        now = time.monotonic()
                        if now - last_progress >= PROGRESS_INTERVAL_SECONDS:
                            print_progress(downloaded, total_size, start_time)
                            last_progress = now

        except (TimeoutError, urllib.error.URLError) as exc:
            retries += 1
            if retries > MAX_DOWNLOAD_RETRIES:
                raise urllib.error.URLError(
                    f"{exc}. Download stopped after {MAX_DOWNLOAD_RETRIES} retries."
                ) from exc
            print(f"  download interrupted ({exc}); retrying {retries}/{MAX_DOWNLOAD_RETRIES}...", flush=True)
            time.sleep(min(2**retries, 30))
            continue

        break

    print_progress(downloaded, total_size, start_time)
    temp_path.replace(destination)


In [31]:
def find_zip_member(zf: zipfile.ZipFile, target_date: dt.date) -> str | None:
    target = target_date.isoformat()
    matches = []
    for member in zf.namelist():
        name = Path(member).name.lower()
        if target in name and name.endswith(".csv"):
            matches.append(member)
    if not matches:
        return None
    if len(matches) > 1:
        raise FileNotFoundError(
            f"Multiple CSV matches for {target} found inside {zf.filename}: {matches}"
        )
    return matches[0]


def output_path_for_date(output_dir: Path, target_date: dt.date) -> Path:
    month_dir = output_dir / target_date.strftime("%Y-%m")
    return month_dir / f"{target_date.isoformat()}_istdaten.csv"


def migrate_legacy_output(output_dir: Path, target_date: dt.date) -> Path | None:
    legacy_path = output_dir / f"{target_date.isoformat()}_istdaten.csv"
    new_path = output_path_for_date(output_dir, target_date)
    if not legacy_path.exists() or new_path.exists():
        return new_path if new_path.exists() else None
    new_path.parent.mkdir(parents=True, exist_ok=True)
    legacy_path.replace(new_path)
    print(f"  moved existing {legacy_path.name} -> {new_path.parent.name}/", flush=True)
    return new_path


def extract_member_with_unzip(archive_path: Path, member: str, output_path: Path) -> None:
    unzip_path = shutil.which("unzip")
    if unzip_path is None:
        raise RuntimeError(
            "Encountered an unsupported ZIP compression method and 'unzip' is not available."
        )
    with output_path.open("wb") as dst:
        result = subprocess.run(
            [unzip_path, "-p", str(archive_path), member],
            stdout=dst,
            stderr=subprocess.PIPE,
            check=False,
        )
    if result.returncode != 0:
        output_path.unlink(missing_ok=True)
        stderr = result.stderr.decode("utf-8", errors="replace").strip()
        raise RuntimeError(
            f"Failed to extract {member} from {archive_path.name} with unzip: {stderr}"
        )


def extract_member(
    zf: zipfile.ZipFile,
    member: str,
    target_date: dt.date,
    output_dir: Path,
    overwrite: bool,
) -> Path:
    migrated = migrate_legacy_output(output_dir, target_date)
    output_path = output_path_for_date(output_dir, target_date)
    if (output_path.exists() or migrated is not None) and not overwrite:
        return output_path

    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with zf.open(member) as src, output_path.open("wb") as dst:
            shutil.copyfileobj(src, dst, length=CHUNK_SIZE)
        return output_path
    except NotImplementedError:
        output_path.unlink(missing_ok=True)
        archive_path = Path(zf.filename) if zf.filename is not None else None
        if archive_path is None:
            raise
        extract_member_with_unzip(archive_path, member, output_path)
        return output_path


In [32]:
def download_selected_istdaten(
    date_ranges: tuple[tuple[str, str], ...] = DEFAULT_SAMPLE_RANGES,
    output_dir: Path = DATA_DIR,
    archive_dir: Path | None = None,
    keep_archives: bool = True,
    overwrite: bool = False,
    dry_run: bool = False,
) -> None:
    dates = expand_date_ranges(date_ranges)
    output_dir = output_dir.resolve()
    archive_dir = (archive_dir or (output_dir / "_archives")).resolve()

    archive_index = parse_archive_index(fetch_text(ARCHIVE_INDEX_URL))
    by_month = group_dates_by_month(dates)
    missing_months = [month for month in by_month if month not in archive_index]
    if missing_months:
        raise FileNotFoundError(
            "No archive ZIP found for these months in the official index: " + ", ".join(sorted(missing_months))
        )

    print(f"Requested {len(dates)} day(s) across {len(by_month)} month archive(s).")
    print(f"Output directory: {output_dir}")
    print(f"Archive directory: {archive_dir}")
    print("Extracted CSV layout: <output-dir>/<YYYY-MM>/<YYYY-MM-DD>_istdaten.csv")

    for month in sorted(by_month):
        archive_urls = archive_index[month]
        month_dates = by_month[month]
        print(f"\n[{month}]")
        print(f"  sources: {len(archive_urls)} archive(s) available")
        preview = ", ".join(date.isoformat() for date in month_dates[:3])
        print("  dates: " + preview + (" ..." if len(month_dates) > 3 else ""))

        if dry_run:
            for archive_url in archive_urls:
                print(f"  source: {archive_url}")
            continue

        remaining_dates = list(month_dates)
        used_archive_paths: list[Path] = []

        for archive_url in archive_urls:
            archive_name = Path(urllib.parse.urlparse(archive_url).path).name
            archive_path = archive_dir / archive_name
            used_archive_paths.append(archive_path)

            print(f"  source: {archive_url}")
            if archive_path.exists():
                print(f"    archive already exists: {archive_path}")
            else:
                print("    downloading archive...")
                download_file(archive_url, archive_path)

            extracted_any = False
            next_remaining: list[dt.date] = []
            with zipfile.ZipFile(archive_path) as zf:
                for target_date in remaining_dates:
                    member = find_zip_member(zf, target_date)
                    # Dates missing from the current ZIP stay in the queue for the next fallback archive.
                    if member is None:
                        next_remaining.append(target_date)
                        continue

                    migrated = migrate_legacy_output(output_dir, target_date)
                    output_path = output_path_for_date(output_dir, target_date)
                    if output_path.exists() and not overwrite:
                        print(f"    skip existing {output_path.parent.name}/{output_path.name}")
                        continue
                    if migrated is not None and not overwrite:
                        print(f"    skip migrated {output_path.parent.name}/{output_path.name}")
                        continue

                    extract_member(zf, member, target_date, output_dir, overwrite)
                    print(f"    extracted {output_path.parent.name}/{output_path.name}")
                    extracted_any = True

            remaining_dates = next_remaining
            if not remaining_dates:
                break
            if not extracted_any:
                print("    archive did not contain requested early dates; trying fallback archive", flush=True)

        if remaining_dates:
            missing = ", ".join(date.isoformat() for date in remaining_dates)
            raise FileNotFoundError(
                f"No archive for {month} contained these requested dates: {missing}"
            )

        if not keep_archives:
            for archive_path in used_archive_paths:
                if archive_path.exists():
                    archive_path.unlink()
                    print(f"  removed archive after extraction: {archive_path.name}")


## 1.4 Raw download configuration

This section decides whether the notebook should reuse the raw daily CSV files that are already present or download the selected months again from the archive.


In [33]:
# Leave this False once the daily CSV extracts already exist under data/YYYY-MM/.
RUN_RAW_DOWNLOAD = False
# The default sample covers four 30-day windows so the project spans multiple seasons.
RAW_DATE_RANGES = DEFAULT_SAMPLE_RANGES
# Keep the downloaded monthly ZIP files unless disk space is more important than reruns.
KEEP_ARCHIVES = True
# Only set this True if you intentionally want to replace previously extracted daily CSVs.
OVERWRITE_RAW_FILES = False

expected_raw_files = expand_date_ranges(RAW_DATE_RANGES)
existing_raw_files = sorted(DATA_DIR.glob("2025-*/*.csv"))
print(f"Existing extracted daily CSV files: {len(existing_raw_files)}")
print(f"Expected files for the default sample: {len(expected_raw_files)}")

if RUN_RAW_DOWNLOAD:
    download_selected_istdaten(
        date_ranges=RAW_DATE_RANGES,
        output_dir=DATA_DIR,
        keep_archives=KEEP_ARCHIVES,
        overwrite=OVERWRITE_RAW_FILES,
    )
else:
    print("Skipping raw data download. Set RUN_RAW_DOWNLOAD = True if the data/YYYY-MM/ folders are missing.")

RAW_FILES = sorted(DATA_DIR.glob("2025-*/*.csv"))
# Stop early with a clear message instead of letting the later filtering steps fail cryptically.
assert RAW_FILES, "No extracted CSV files found under data/YYYY-MM/. Set RUN_RAW_DOWNLOAD = True to download them."

RG_PATH = shutil.which("rg")
if RG_PATH is None:
    print(
        "ripgrep was not found. Falling back to pure Python row scanning. "
        "This is slower, but it produces the same downstream filtered dataset."
    )
else:
    print(f"Using ripgrep for faster preliminary filtering: {RG_PATH}")

with RAW_FILES[0].open("r", encoding="utf-8-sig") as fh:
    HEADER_LINE = fh.readline()

print(f"Daily CSV files found: {len(RAW_FILES)}")
print(f"First file: {RAW_FILES[0].relative_to(ROOT)}")


Existing extracted daily CSV files: 120
Expected files for the default sample: 120
Skipping raw data download. Set RUN_RAW_DOWNLOAD = True if the data/YYYY-MM/ folders are missing.
ripgrep was not found. Falling back to pure Python row scanning. This is slower, but it produces the same downstream filtered dataset.
Daily CSV files found: 120
First file: data/2025-01/2025-01-01_istdaten.csv
